# Experiment 03
## Performance Analysis of Activation Functions: (Case Study)

**Aim:** To study, compare, and analyze the performance of fundamental activation functions:
1. **ReLU (Rectified Linear Unit)**
2. **Sigmoid**
3. **Tanh (Hyperbolic Tangent)**
4. **Softmax**

**Objectives:**
- Mathematically define and visualize all four activation functions.
- Train identical neural networks with ReLU, Sigmoid, and Tanh to observe training convergence and accuracy.
- Understand the role of Softmax in output probability calculation.
- Document a detailed Case Study (Vanishing Gradient, computational efficiency, and use cases).

### Cell 1: Import Required Libraries

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import time

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models

print("TensorFlow Version:", tf.__version__)
print("Libraries loaded successfully!")

### Cell 2: Mathematical Visualization of Activation Functions

We plot the mathematical curves for **Sigmoid, Tanh, ReLU**, and demonstrate **Softmax** probability distribution.

In [ ]:
# Input range for plotting
x = np.linspace(-6, 6, 200)

# 1. Sigmoid Function: 1 / (1 + e^-x)
sigmoid = 1 / (1 + np.exp(-x))

# 2. Tanh Function: (e^x - e^-x) / (e^x + e^-x)
tanh = np.tanh(x)

# 3. ReLU Function: max(0, x)
relu = np.maximum(0, x)

# 4. Softmax Function (Applied to sample logits)
sample_logits = np.array([2.0, 1.0, 0.1, 3.5, 0.8])
softmax_probs = np.exp(sample_logits) / np.sum(np.exp(sample_logits))

# Plotting all 4
plt.figure(figsize=(14, 8))

# Sigmoid Plot
plt.subplot(2, 2, 1)
plt.plot(x, sigmoid, color='blue', lw=2)
plt.title("1. Sigmoid Function [Range: (0, 1)]")
plt.grid(True)
plt.axhline(0, color='black', lw=0.5)
plt.axvline(0, color='black', lw=0.5)

# Tanh Plot
plt.subplot(2, 2, 2)
plt.plot(x, tanh, color='green', lw=2)
plt.title("2. Tanh Function [Range: (-1, 1)] (Zero-Centered)")
plt.grid(True)
plt.axhline(0, color='black', lw=0.5)
plt.axvline(0, color='black', lw=0.5)

# ReLU Plot
plt.subplot(2, 2, 3)
plt.plot(x, relu, color='red', lw=2)
plt.title("3. ReLU Function [Range: [0, ∞)] (Non-Saturating)")
plt.grid(True)
plt.axhline(0, color='black', lw=0.5)
plt.axvline(0, color='black', lw=0.5)

# Softmax Plot
plt.subplot(2, 2, 4)
classes = [f'Class {i}' for i in range(len(sample_logits))]
plt.bar(classes, softmax_probs, color='purple', alpha=0.7)
plt.title("4. Softmax Probabilities [Sum = 1.0]")
plt.ylabel("Probability")
plt.grid(axis='y', linestyle='--', alpha=0.7)

plt.suptitle("Mathematical Curves of Core Activation Functions", fontsize=15)
plt.tight_layout()
plt.show()

### Cell 3: Load Dataset for Performance Benchmark (MNIST)

We use a fast 5,000-sample subset of MNIST handwritten digits to benchmark training speed and convergence across activation functions.

In [ ]:
(X_train_full, y_train_full), (X_test_full, y_test_full) = keras.datasets.mnist.load_data()

# Normalization
X_train = (X_train_full[:5000] / 255.0)
y_train = y_train_full[:5000]

X_test = (X_test_full[:1000] / 255.0)
y_test = y_test_full[:1000]

print(f"Benchmark Training Data: {X_train.shape}")
print(f"Benchmark Testing Data: {X_test.shape}")

### Cell 4: Function to Build and Train Identical Neural Networks

Every network has identical architecture and hyperparameters, differing **only** in the hidden layer activation function (`relu`, `sigmoid`, or `tanh`).
The output layer uses **Softmax** for 10-class probability distribution.

In [ ]:
def train_model(activation_func, epochs=10):
    print(f"Training model with: {activation_func.upper()}...")
    
    model = models.Sequential([
        layers.Input(shape=(28, 28)),
        layers.Flatten(),
        layers.Dense(128, activation=activation_func),
        layers.Dense(64, activation=activation_func),
        layers.Dense(10, activation='softmax')  # Output layer uses Softmax
    ])
    
    model.compile(
        optimizer='adam',
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    
    start_time = time.time()
    history = model.fit(
        X_train, y_train,
        epochs=epochs,
        batch_size=64,
        validation_data=(X_test, y_test),
        verbose=0
    )
    elapsed_time = time.time() - start_time
    
    test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
    print(f"Finished in {elapsed_time:.2f}s | Test Accuracy: {test_acc*100:.2f}%")
    
    return history, test_acc, elapsed_time

### Cell 5: Training and Benchmarking (ReLU vs Sigmoid vs Tanh)

In [ ]:
# Training models for 10 epochs each
history_relu, acc_relu, time_relu = train_model('relu')
history_tanh, acc_tanh, time_tanh = train_model('tanh')
history_sigmoid, acc_sigmoid, time_sigmoid = train_model('sigmoid')

### Cell 6: Performance Analysis Curves (Loss and Accuracy Comparison)

Here, we plot the comparative learning trajectories side by side.

In [ ]:
epochs_range = range(1, 11)
plt.figure(figsize=(14, 5))

# 1. Validation Accuracy Comparison
plt.subplot(1, 2, 1)
plt.plot(epochs_range, history_relu.history['val_accuracy'], 'r-o', label=f"ReLU (Final: {acc_relu*100:.1f}%)", lw=2)
plt.plot(epochs_range, history_tanh.history['val_accuracy'], 'g-s', label=f"Tanh (Final: {acc_tanh*100:.1f}%)", lw=2)
plt.plot(epochs_range, history_sigmoid.history['val_accuracy'], 'b-^', label=f"Sigmoid (Final: {acc_sigmoid*100:.1f}%)", lw=2)
plt.title('Validation Accuracy Comparison over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Validation Accuracy')
plt.legend()
plt.grid(True)

# 2. Training Loss Comparison
plt.subplot(1, 2, 2)
plt.plot(epochs_range, history_relu.history['loss'], 'r-o', label='ReLU Loss', lw=2)
plt.plot(epochs_range, history_tanh.history['loss'], 'g-s', label='Tanh Loss', lw=2)
plt.plot(epochs_range, history_sigmoid.history['loss'], 'b-^', label='Sigmoid Loss', lw=2)
plt.title('Training Loss Convergence Comparison')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

### Cell 7: Performance Summary Table

In [ ]:
summary_data = {
    'Activation Function': ['ReLU', 'Tanh', 'Sigmoid', 'Softmax'],
    'Output Range': ['[0, ∞)', '(-1, 1)', '(0, 1)', '(0, 1) [Sum=1]'],
    'Convergence Speed': ['Fastest (No Vanishing Gradient)', 'Moderate (Zero-centered)', 'Slowest (Vanishing Gradient)', 'N/A (Used at Output)'],
    'Test Accuracy': [f"{acc_relu*100:.2f}%", f"{acc_tanh*100:.2f}%", f"{acc_sigmoid*100:.2f}%", 'Integrated in Output Layer'],
    'Primary Usage': ['Hidden layers of Deep Nets', 'Hidden layers (shallow nets/RNNs)', 'Binary output layer', 'Multi-class output layer']
}

summary_df = pd.DataFrame(summary_data)
summary_df

### Cell 8: Detailed Case Study (Theory & Analytical Observations)

#### **1. ReLU (Rectified Linear Unit)**
- **Formula:** $f(x) = \max(0, x)$
- **Derivative:** $f'(x) = 1$ for $x > 0$, and $0$ for $x < 0$.
- **Key Advantage:** Constant gradient of $1$ for positive inputs completely avoids the **Vanishing Gradient Problem**, enabling very fast training in deep networks.
- **Limitation (Dying ReLU):** If an entire batch yields negative inputs, the neuron outputs $0$ and stops learning. Variants like *Leaky ReLU* solve this.

---

#### **2. Sigmoid Activation**
- **Formula:** $\sigma(x) = \frac{1}{1 + e^{-x}}$
- **Derivative:** $\sigma'(x) = \sigma(x)(1 - \sigma(x))$, with a maximum value of only $0.25$!
- **Vanishing Gradient Problem:** When multiplying many small gradients ($< 0.25$) during backpropagation through deep layers, gradients approach $0$, causing weights in early layers to stop updating.
- **Primary Use Case:** Only in the final layer for **Binary Classification** (outputting probability between 0 and 1).

---

#### **3. Tanh (Hyperbolic Tangent)**
- **Formula:** $\tanh(x) = \frac{e^x - e^{-x}}{e^x + e^{-x}}$
- **Range:** $(-1, 1)$ — **Zero-Centered**.
- **Advantage over Sigmoid:** Being zero-centered makes gradient updates more balanced, leading to faster convergence than Sigmoid.
- **Limitation:** Still suffers from gradient saturation/vanishing gradients at high and low extreme inputs.

---

#### **4. Softmax Activation**
- **Formula:** $\text{Softmax}(z_i) = \frac{e^{z_i}}{\sum_{j=1}^{K} e^{z_j}}$
- **Role:** Normalizes unconstrained output scores (logits) into a valid categorical probability distribution where all probabilities sum strictly to $1.0$.
- **Usage:** Exclusively used in the **Final Layer of Multi-Class Classification** networks (e.g., MNIST 10-digit classification).

## Result and Conclusion

1. Plotted mathematical curves and analyzed characteristics of **ReLU, Sigmoid, Tanh**, and **Softmax**.
2. Evaluated training performance under identical architectures:
   - **ReLU** achieved the fastest loss reduction and highest test accuracy.
   - **Tanh** performed better than Sigmoid due to zero-centered outputs.
   - **Sigmoid** showed the slowest convergence rate due to small derivative values.
   - **Softmax** effectively converted output logits into 10-class probabilities.